In [7]:
import requests
import numpy as np
import pandas as pd
import sqlite3
from datetime import datetime, timezone


Create tables

In [12]:
conn = sqlite3.connect("screener.db")
cursor = conn.cursor()

cursor.execute("""
    CREATE TABLE IF NOT EXISTS pools (
    pair_address TEXT PRIMARY KEY,
    token_address TEXT,
    chain_id TEXT,
    symbol TEXT,
    dex_id TEXT,
    price_usd REAL,
    liquidity_usd REAL,
    volume_m5 REAL,
    volume_h1 REAL,
    volume_h24 REAL,
    price_change_m5 REAL,
    price_change_h1 REAL,
    price_change_h24 REAL,
    market_cap REAL,
    fdv REAL,
    pair_created_at INTEGER,
    first_seen_at TIMESTAMP,
    last_updated_at TIMESTAMP
)
""")

cursor.execute("""
    CREATE TABLE IF NOT EXISTS pool_snapshots (
        id               INTEGER PRIMARY KEY AUTOINCREMENT,
        pair_address     TEXT REFERENCES pools(pair_address),
        price_usd        REAL,
        liquidity_usd    REAL,
        volume_m5        REAL,
        volume_h1        REAL,
        volume_h24       REAL,
        price_change_m5  REAL,
        price_change_h1  REAL,
        price_change_h24 REAL,
        market_cap       REAL,
        fdv              REAL,
        snapshot_at      TIMESTAMP
    )
""")
conn.commit()

In [13]:
response = requests.get("https://api.dexscreener.com/token-profiles/latest/v1", headers={"Accept": "*/*"})
tokens = response.json()
all_pairs = []
new_tokens_count = 0

for token in tokens:
    chain_id = token.get("chainId")
    token_address = token.get("tokenAddress")

    cursor.execute(
        """SELECT 1 FROM pools WHERE token_address = ? """,
        (token_address,)
    )

    if cursor.fetchone():
        continue

    print(f"New token discovered: {token_address}")
    new_tokens_count += 1

    now = datetime.now(timezone.utc).isoformat()
    data = requests.get(f"https://api.dexscreener.com/token-pairs/v1/{chain_id}/{token_address}",headers={"Accept": "*/*"})
    pair_response = data.json()

    pair_details = [{ "token_address" : token_address, 
                     "symbol" : item.get('baseToken').get('symbol'),
                     "pair_address" : item.get('pairAddress'),
                     "dex_id" : item.get('dexId'),
                     "price_usd" : item.get('priceUsd'),
                     "price_change_m5" : item.get('priceChange', {}).get('m5'),
                     "price_change_h1" : item.get('priceChange', {}).get('h1'),
                     "price_change_h24" : item.get('priceChange', {}).get('h24'),
                     "liquidity_usd" : item.get('liquidity', {}).get('usd'),
                     "volume_m5" : item.get('volume', {}).get('m5'),
                     "volume_h1" : item.get('volume', {}).get('h1'),
                     "volume_h24" : item.get('volume', {}).get('h24'),
                     "market_cap" : item.get('marketCap'),
                     "fdv" : item.get('fdv'),
                     "pair_created_at" : item.get('pairCreatedAt'),
                     "timestamp_fetched" : now,
    }
    for item in pair_response ]
    all_pairs.extend(pair_details)

    for pool in pair_details:
        cursor.execute(
            """
            INSERT INTO pools (
                pair_address,
                token_address,
                chain_id,
                symbol,
                dex_id,
                price_usd,
                liquidity_usd,
                volume_m5,
                volume_h1,
                volume_h24,
                price_change_m5,
                price_change_h1,
                price_change_h24,
                market_cap,
                fdv,
                pair_created_at,
                first_seen_at,
                last_updated_at
            )
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?) ON CONFLICT(pair_address) DO UPDATE SET
                price_usd       = excluded.price_usd,
                liquidity_usd   = excluded.liquidity_usd,
                volume_m5       = excluded.volume_m5,
                volume_h1       = excluded.volume_h1,
                volume_h24      = excluded.volume_h24,
                price_change_m5  = excluded.price_change_m5,
                price_change_h1  = excluded.price_change_h1,
                price_change_h24 = excluded.price_change_h24,
                market_cap      = excluded.market_cap,
                fdv             = excluded.fdv,
                last_updated_at = excluded.last_updated_at
            """,
            (
                pool["pair_address"],
                pool["token_address"],
                chain_id,
                pool["symbol"],
                pool["dex_id"],
                pool["price_usd"],
                pool["liquidity_usd"],
                pool["volume_m5"],
                pool["volume_h1"],
                pool["volume_h24"],
                pool["price_change_m5"],
                pool["price_change_h1"],
                pool["price_change_h24"],
                pool["market_cap"],
                pool["fdv"],
                pool["pair_created_at"],
                now,
                now
            )
        )
conn.commit()

df = pd.DataFrame(all_pairs)
print(f"New tokens discovered: {new_tokens_count}")
print(f"Pairs collected: {len(df)}")
conn.close()

New token discovered: 0xA4acBCf08968397bA5fdAa2c45Dd4822B340eD5d
New token discovered: F4fYLCbcQaDs5dHjF9qhG5sq5mKV2rauiZUYjuWApump
New token discovered: dUaSdZm38CStjhp1VSJWGFQqSxf6DQB2hfjMXmVpump
New token discovered: 797qii8caT4cdXnywtqEBzk4c7kN8URGUwSYG8b4pump
New token discovered: EsTkwKZKpZ9Dz54mbDWumfsBx3L8e73YmghLJFQhpump
New token discovered: 5SYEutBJ3uVwWu5KbM7pcPHH7ZsRU4sqaMe66cMmpump
New token discovered: 7vTSYWLNyZJs3XMsMc5bB5HpTshjXmDe9gCjQFXKpump
New token discovered: GLwHaEdjsLp83CgjuxxuSvqS2z91t6oJmec1mSu4pump
New token discovered: GF3upFzJJPtJzEXTPFL8Hf23zToqLwFCbuLs7qNBpump
New token discovered: 0xf2da4643B573042ce685BAa36a855A8812e5CF31
New token discovered: 7xUvzbjxSqpK36fP6Nuqo6EerhLUewuEEB8vFsZDpump
New token discovered: 9gXef7aV6TnTDnsiPE9qTQzaLA1zH11oRoPFB9rMpump
New token discovered: 5jCQH9EwoqbSNs3ZHWE2b4e6x7PpecYfFuhgR3uRpump
New token discovered: v8sssEryFSYa5VFdcsq6Y1gBxQ71XaMs2DVPhBDpump
New token discovered: 8ycCZqdfr8rJv61rwB7WCGHv8iZbXp6FPCANgwDapump
N

In [14]:
conn = sqlite3.connect("screener.db")
cursor = conn.cursor()

cursor.execute("""
    SELECT pair_address, token_address, chain_id
    FROM pools
""")
pools = cursor.fetchall()
snapshot_count = 0

for pair_address, token_address, chain_id in pools:

    response = requests.get(
        f"https://api.dexscreener.com/token-pairs/v1/{chain_id}/{token_address}"
    )

    pair_response = response.json()
    pool = next(
        (
            p
            for p in pair_response
            if p.get("pairAddress") == pair_address
        ),
        None
    )

    if pool is None:
        continue

    now = datetime.now(timezone.utc).isoformat()

    cursor.execute(
        """
        INSERT INTO pool_snapshots(
            pair_address,
            price_usd,
            liquidity_usd,
            volume_m5,
            volume_h1,
            volume_h24,
            price_change_m5,
            price_change_h1,
            price_change_h24,
            market_cap,
            fdv,
            snapshot_at
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (
            pair_address,
            pool.get("priceUsd"),
            pool.get("liquidity", {}).get("usd"),
            pool.get("volume", {}).get("m5"),
            pool.get("volume", {}).get("h1"),
            pool.get("volume", {}).get("h24"),
            pool.get("priceChange", {}).get("m5"),
            pool.get("priceChange", {}).get("h1"),
            pool.get("priceChange", {}).get("h24"),
            pool.get("marketCap"),
            pool.get("fdv"),
            now,
        ),
    )
    snapshot_count += 1

conn.commit()
print(f"Snapshots inserted: {snapshot_count}")
conn.close()

Snapshots inserted: 83


In [15]:
%reload_ext sql
%sql sqlite:////Users/ismail/GitHub_Projects/Solana-dexscreener_bot/screener.db

In [17]:
%%sql
SELECT pair_address,
       COUNT(*)
FROM pool_snapshots
GROUP BY pair_address
ORDER BY COUNT(*) DESC;

Running query in 'sqlite:////Users/ismail/GitHub_Projects/Solana-dexscreener_bot/screener.db'

pair_address,COUNT(*)
hEpaioScfe2HozdSsqSPa6eTwYnYfPYDVmgS7SiWrPk,1
evy8pdEVXUVzDH7JYLktasFSiHyBvU5VPpKUhdFkADi,1
UxWzxNptkAZzTDuk8tSj7GAjLPgjL1wdYCFu35CjGA1,1
TUuXcTwvjnbMpV1kykPYxKapimjkQ875PjCtkokffjv,1
HWuNK5xEdiujKp6MZyNTYVkwzZeUxC7PQDGuMNVytybR,1
HLVDbreM66AfaUrEisZHfxFsS3kKvS51Ezv3r86H7D7n,1
H8UxevknB9wMutM9fxy5ihrVLtVqssb6bcHwFiTWkcxe,1
H5FCHAdHw5LZ1w76HhTFwNUyv9a5hjVBZ9WqrhphZyHn,1
Gn92PsVsetsZc8KX7DNQ2VetmjfCrqPxJ7LfH7gwUCEX,1
GJA8ec4mGb79zP3cNiz92VDoZP4KHCSQPJ1xKNKaBHW1,1
